# Day 7 总目标

今天你要实现：

Markdown documents
↓
load_all_documents()
↓
Document 列表
↓
RecursiveCharacterTextSplitter 切分
↓
Embedding 模型向量化
↓
Chroma 本地持久化
↓
storage/chroma_db

今天主要新增 / 修改这些文件：

F:\ResearchAgent
├── src
│   └── research_agent
│       └── rag
│           ├── indexer.py
│           └── loaders.py  # 已有，不一定改
│
├── scripts
│   ├── build_index.py
│   └── test_chroma_search.py
│
├── storage
│   └── chroma_db
│
└── requirements.txt

# 二、更新 requirements.txt

在 requirements.txt 里加入：

langchain-chroma
chromadb
langchain-text-splitters
langchain-huggingface
sentence-transformers

你可以保留原来的内容，最终大概是：

langgraph
langchain
langchain-core
langchain-openai
langchain-chroma
chromadb
langchain-text-splitters
langchain-huggingface
sentence-transformers
python-dotenv
pydantic
rich
tqdm
ipykernel

# 三、更新 .gitignore

打开：

F:\ResearchAgent\.gitignore

确认加入：

# Vector database
storage/

原因是：

storage/chroma_db 是本地生成的向量库
体积可能变大
不适合上传 GitHub
别人可以通过 scripts/build_index.py 自己重建

# 四、创建 indexer.py

路径：

F:\ResearchAgent\src\research_agent\rag\indexer.py

复制下面代码：

from pathlib import Path
import shutil
from typing import List

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from .loaders import load_all_documents


PROJECT_ROOT = Path(__file__).resolve().parents[3]
STORAGE_DIR = PROJECT_ROOT / "storage"
CHROMA_DIR = STORAGE_DIR / "chroma_db"

COLLECTION_NAME = "research_agent_docs"

DEFAULT_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"


def get_embedding_model(model_name: str = DEFAULT_EMBEDDING_MODEL):
    """
    创建 embedding 模型。

    Day 7 默认使用本地 sentence-transformers 模型，
    避免依赖 API key。
    """
    return HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )


def split_documents(
    documents: List[Document],
    chunk_size: int = 500,
    chunk_overlap: int = 80,
) -> List[Document]:
    """
    将原始 Document 切分成更小的 chunk。

    chunk_size:
        每个片段的大致长度。

    chunk_overlap:
        相邻片段之间保留的重叠长度，避免上下文断裂。
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", "。", "，", " ", ""],
    )

    chunks = splitter.split_documents(documents)

    for i, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = i

    return chunks


def reset_vector_index(persist_directory: Path = CHROMA_DIR) -> None:
    """
    删除旧的 Chroma 向量库目录。
    """
    if persist_directory.exists():
        shutil.rmtree(persist_directory)


def build_vector_index(
    reset: bool = True,
    persist_directory: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
) -> Chroma:
    """
    构建 Chroma 向量索引。

    流程：
    1. 加载 markdown Documents
    2. 切分成 chunks
    3. 创建 embedding 模型
    4. 写入 Chroma
    5. 持久化到 storage/chroma_db
    """
    if reset:
        reset_vector_index(persist_directory)

    documents = load_all_documents()
    print(f"Loaded {len(documents)} documents.")

    if not documents:
        raise ValueError("No documents loaded. Please check the data directory.")

    chunks = split_documents(documents)
    print(f"Split into {len(chunks)} chunks.")

    embedding_model = get_embedding_model()
    print(f"Using embedding model: {DEFAULT_EMBEDDING_MODEL}")

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=str(persist_directory),
        collection_name=collection_name,
    )

    print(f"Chroma index saved to: {persist_directory}")

    return vector_store


def load_vector_store(
    persist_directory: Path = CHROMA_DIR,
    collection_name: str = COLLECTION_NAME,
) -> Chroma:
    """
    加载已经构建好的 Chroma 向量库。
    """
    if not persist_directory.exists():
        raise FileNotFoundError(
            f"Chroma index not found at {persist_directory}. "
            f"Please run scripts/build_index.py first."
        )

    embedding_model = get_embedding_model()

    vector_store = Chroma(
        persist_directory=str(persist_directory),
        collection_name=collection_name,
        embedding_function=embedding_model,
    )

    return vector_store


def similarity_search(
    query: str,
    k: int = 3,
) -> List[Document]:
    """
    最小检索测试函数。
    """
    vector_store = load_vector_store()
    return vector_store.similarity_search(query, k=k)